# Movie Recommendation System

## Movie Recommendation System Plan

To build a movie recommendation system, we'll generally follow these steps:

1.  **Data Loading and Initial Exploration:** Load the necessary datasets (e.g., movies, ratings, user data) and perform initial checks.
2.  **Data Preprocessing:** Clean and prepare the data for analysis and modeling. This might include handling missing values, merging datasets, and feature engineering.
3.  **Feature Engineering (Optional but Recommended):** Create new features that might be useful for the recommendation system.
4.  **Model Selection:** Choose an appropriate recommendation algorithm (e.g., collaborative filtering, content-based filtering, hybrid approaches).
5.  **Model Training:** Train the chosen model on the prepared data.
6.  **Model Evaluation:** Evaluate the performance of the recommendation system using relevant metrics.
7.  **Recommendation Generation:** Implement a function to generate recommendations for a given user or based on a movie.

Let's start by loading the data. We'll typically need movie data (titles, genres, etc.) and user ratings.

### Data Loading: MovieLens 100k Dataset

We will use the MovieLens 100k dataset for this recommendation system. This dataset contains 100,000 ratings from 943 users on 1,682 movies.

In [14]:
import pandas as pd
import requests
import zipfile
import io
import os

# Define the URL for the MovieLens 100k dataset
MOVIES_URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
DATA_DIR = './ml-100k'

# Download and extract the dataset if not already present
if not os.path.exists(DATA_DIR):
    print("Downloading MovieLens 100k dataset...")
    response = requests.get(MOVIES_URL)
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall('.')
    print("Download and extraction complete.")
else:
    print("MovieLens 100k dataset already exists.")

# Define column names based on the dataset's README
# For u.data (ratings file)
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'u.data'), sep='\t', names=ratings_cols, encoding='latin-1')

# For u.item (movies file)
# The 'genres' are 19 binary columns, followed by the others
movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL',
              'unknown', 'Action', 'Adventure', 'Animation', 'Children\'s', 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies_df = pd.read_csv(os.path.join(DATA_DIR, 'u.item'), sep='|', names=movie_cols, encoding='latin-1')

print("Ratings DataFrame head:")
display(ratings_df.head())

print("\nMovies DataFrame head:")
display(movies_df.head())

MovieLens 100k dataset already exists.
Ratings DataFrame head:


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596



Movies DataFrame head:


,movie_id,movie_title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### Data Preprocessing: Merging DataFrames and Cleaning

To simplify our dataset and make it ready for analysis, we'll merge the `ratings_df` and `movies_df` on `movie_id`. We'll also drop some columns that are not directly used in a basic collaborative filtering recommendation system, such as `timestamp`, `release_date`, `video_release_date`, and `IMDb_URL`.

In [15]:
# Drop unnecessary columns from movies_df
movies_df_cleaned = movies_df.drop(columns=['release_date', 'video_release_date', 'IMDb_URL'])

# Drop the 'timestamp' column from ratings_df as it's not needed for a basic recommendation system
ratings_df = ratings_df.drop(columns=['timestamp'])

# Merge the ratings and movies DataFrames
merged_df = pd.merge(ratings_df, movies_df_cleaned, on='movie_id')

print("Merged DataFrame head:")
display(merged_df.head())

print("\nMerged DataFrame info:")
merged_df.info()

Merged DataFrame head:


,user_id,movie_id,rating,movie_title,unknown,Action,Adventure,Animation,Children's,Comedy,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,196,242,3,Kolya (1996),0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,186,302,3,L.A. Confidential (1997),0,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
2,22,377,1,Heavyweights (1994),0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
3,244,51,2,Legends of the Fall (1994),0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,1
4,166,346,1,Jackie Brown (1997),0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Merged DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   user_id      100000 non-null  int64 
 1   movie_id     100000 non-null  int64 
 2   rating       100000 non-null  int64 
 3   movie_title  100000 non-null  object
 4   unknown      100000 non-null  int64 
 5   Action       100000 non-null  int64 
 6   Adventure    100000 non-null  int64 
 7   Animation    100000 non-null  int64 
 8   Children's   100000 non-null  int64 
 9   Comedy       100000 non-null  int64 
 10  Crime        100000 non-null  int64 
 11  Documentary  100000 non-null  int64 
 12  Drama        100000 non-null  int64 
 13  Fantasy      100000 non-null  int64 
 14  Film-Noir    100000 non-null  int64 
 15  Horror       100000 non-null  int64 
 16  Musical      100000 non-null  int64 
 17  Mystery      100000 non-null  int64 
 18  Romance      100000 n

### Creating the User-Item Matrix

For collaborative filtering, it's often useful to represent the data as a user-item matrix, where rows are users, columns are movies, and values are ratings. We'll use the `pivot_table` function from pandas to create this matrix.

In [16]:
# Create the user-item matrix
user_movie_matrix = merged_df.pivot_table(index='user_id', columns='movie_title', values='rating')

print("User-Item Matrix head:")
display(user_movie_matrix.head())

print("\nUser-Item Matrix shape:", user_movie_matrix.shape)

User-Item Matrix head:


movie_title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN



User-Item Matrix shape: (943, 1664)


### Building the Recommendation Function

Now, let's create a function that, given a movie title, recommends other movies. We'll achieve this by calculating the correlation between the ratings of the input movie and all other movies in our `user_movie_matrix`. A higher correlation indicates more similar rating patterns from users, suggesting similar movie preferences.

In [17]:
def recommend_movies(movie_title, user_movie_matrix, num_recommendations=10):
    """
    Recommends movies similar to the given movie title based on user ratings correlation.

    Args:
        movie_title (str): The title of the movie to find recommendations for.
        user_movie_matrix (pd.DataFrame): The user-item matrix with users as rows,
                                         movies as columns, and ratings as values.
        num_recommendations (int): The number of top similar movies to recommend.

    Returns:
        pd.Series: A Series of recommended movie titles with their correlation scores.
                   Returns an empty Series if the movie is not found.
    """
    try:
        # Get the ratings for the target movie
        movie_ratings = user_movie_matrix[movie_title]
    except KeyError:
        print(f"Movie '{movie_title}' not found in the dataset.")
        return pd.Series()

    # Calculate correlation with all other movies
    similar_to_movie = user_movie_matrix.corrwith(movie_ratings)

    # Convert to DataFrame and remove NaN values (movies with no common raters)
    corr_movie = pd.DataFrame(similar_to_movie, columns=['Correlation'])
    corr_movie.dropna(inplace=True)

    # Filter out movies that have less than a certain number of ratings
    # To do this, we first need to count how many users rated each movie.
    # We can get this by counting non-NaN values in each column of user_movie_matrix.
    movie_ratings_count = user_movie_matrix.count()
    # Convert to DataFrame for easier merging/filtering
    movie_ratings_count_df = pd.DataFrame(movie_ratings_count, columns=['RatingCount'])

    # Merge the correlation with the rating counts
    corr_movie = corr_movie.merge(movie_ratings_count_df, left_index=True, right_index=True)

    # Set a threshold for minimum number of ratings
    min_ratings_threshold = 100 # A common threshold for MovieLens 100k
    corr_movie = corr_movie[corr_movie['RatingCount'] >= min_ratings_threshold]

    # Sort the movies by correlation in descending order
    # and exclude the movie itself from recommendations
    recommendations = corr_movie.sort_values('Correlation', ascending=False)
    recommendations = recommendations[recommendations.index != movie_title]

    return recommendations.head(num_recommendations)

# Example usage:
print("Recommendations for 'Star Wars (1977)':")
display(recommend_movies('Star Wars (1977)', user_movie_matrix))

print("\nRecommendations for 'Toy Story (1995)':")
display(recommend_movies('Toy Story (1995)', user_movie_matrix))

Recommendations for 'Star Wars (1977)':


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,Correlation,RatingCount
movie_title,,
"Empire Strikes Back, The (1980)",0.747981,367
Return of the Jedi (1983),0.672556,507
Raiders of the Lost Ark (1981),0.536117,420
Austin Powers: International Man of Mystery (1997),0.377433,130
"Sting, The (1973)",0.367538,241
Indiana Jones and the Last Crusade (1989),0.350107,331
Pinocchio (1940),0.347868,101
"Frighteners, The (1996)",0.332729,115
L.A. Confidential (1997),0.319065,297



Recommendations for 'Toy Story (1995)':


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,Correlation,RatingCount
movie_title,,
"Craft, The (1996)",0.549100,104
Down Periscope (1996),0.457995,101
Miracle on 34th Street (1994),0.456291,101
G.I. Jane (1997),0.454756,175
Amistad (1997),0.449915,124
Beauty and the Beast (1991),0.442960,202
"Mask, The (1994)",0.432855,129
Cinderella (1950),0.428372,129
That Thing You Do! (1996),0.427936,176


### Creating a Flask API for Recommendations

To make our recommendation system accessible, we can wrap the `recommend_movies` function in a simple web API using Flask. This will allow us to send a movie title to the API and receive recommendations as a response.

First, let's create a file named `app.py` that will contain our Flask application. We'll also need to install Flask.

In [18]:
!pip install Flask

In [19]:
%%writefile app.py

from flask import Flask, request, jsonify
import pandas as pd
import os
import zipfile
import io
import requests

app = Flask(__name__)

# --- Data Loading and Preprocessing (Copied from previous steps) ---
# Define the URL for the MovieLens 100k dataset
MOVIES_URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
DATA_DIR = './ml-100k'

# Download and extract the dataset if not already present
if not os.path.exists(DATA_DIR):
    print("Downloading MovieLens 100k dataset...")
    response = requests.get(MOVIES_URL)
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall('.')
    print("Download and extraction complete.")

# Define column names based on the dataset's README
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'u.data'), sep='\t', names=ratings_cols, encoding='latin-1')

movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL',
              'unknown', 'Action', 'Adventure', 'Animation', 'Children\'s', 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies_df = pd.read_csv(os.path.join(DATA_DIR, 'u.item'), sep='|', names=movie_cols, encoding='latin-1')

# Drop unnecessary columns and merge
movies_df_cleaned = movies_df.drop(columns=['release_date', 'video_release_date', 'IMDb_URL'])
ratings_df = ratings_df.drop(columns=['timestamp'])
merged_df = pd.merge(ratings_df, movies_df_cleaned, on='movie_id')

# Create user-item matrix
user_movie_matrix = merged_df.pivot_table(index='user_id', columns='movie_title', values='rating')

# --- Recommendation Function (Copied from previous steps) ---
def recommend_movies(movie_title, user_movie_matrix, num_recommendations=10):
    try:
        movie_ratings = user_movie_matrix[movie_title]
    except KeyError:
        return pd.Series(dtype=float) # Return empty series for consistency

    similar_to_movie = user_movie_matrix.corrwith(movie_ratings)
    corr_movie = pd.DataFrame(similar_to_movie, columns=['Correlation'])
    corr_movie.dropna(inplace=True)

    movie_ratings_count = user_movie_matrix.count()
    movie_ratings_count_df = pd.DataFrame(movie_ratings_count, columns=['RatingCount'])
    corr_movie = corr_movie.merge(movie_ratings_count_df, left_index=True, right_index=True)

    min_ratings_threshold = 100
    corr_movie = corr_movie[corr_movie['RatingCount'] >= min_ratings_threshold]

    recommendations = corr_movie.sort_values('Correlation', ascending=False)
    recommendations = recommendations[recommendations.index != movie_title]

    return recommendations.head(num_recommendations)

# --- Flask Routes ---
@app.route('/recommend', methods=['GET'])
def get_recommendations():
    movie_title = request.args.get('movie')
    if not movie_title:
        return jsonify({'error': 'Please provide a movie title as a query parameter (e.g., /recommend?movie=Toy Story (1995))'}), 400

    recommendations = recommend_movies(movie_title, user_movie_matrix)

    if recommendations.empty:
        return jsonify({'message': f"No recommendations found for '{movie_title}'. It might not be in the dataset or have enough ratings."}) , 404
    else:
        return jsonify(recommendations.to_dict(orient='index'))

@app.route('/')
def home():
    return "<h1>Movie Recommendation API</h1><p>Use /recommend?movie=<movie_title> to get recommendations.</p>"

if __name__ == '__main__':
    # In a Colab environment, we might need a specific host or port for external access
    # For local testing within Colab, '127.0.0.1' is fine.
    # If deploying, you'd configure this differently.
    app.run(host='127.0.0.1', port=5000)

Overwriting app.py


In [31]:
%%writefile app.py

from flask import Flask, request, jsonify
import pandas as pd
import os
import zipfile
import io
import requests

app = Flask(__name__)

# --- Data Loading and Preprocessing (Copied from previous steps) ---
# Define the URL for the MovieLens 100k dataset
MOVIES_URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
DATA_DIR = './ml-100k'

# Download and extract the dataset if not already present
if not os.path.exists(DATA_DIR):
    # print("Downloading MovieLens 100k dataset...") # Removed print statement
    response = requests.get(MOVIES_URL)
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall('.')
    # print("Download and extraction complete.") # Removed print statement

# Define column names based on the dataset's README
ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'u.data'), sep='\t', names=ratings_cols, encoding='latin-1')

movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL',
              'unknown', 'Action', 'Adventure', 'Animation', 'Children\'s', 'Comedy',
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies_df = pd.read_csv(os.path.join(DATA_DIR, 'u.item'), sep='|', names=movie_cols, encoding='latin-1')

# Drop unnecessary columns and merge
movies_df_cleaned = movies_df.drop(columns=['release_date', 'video_release_date', 'IMDb_URL'])
ratings_df = ratings_df.drop(columns=['timestamp'])
merged_df = pd.merge(ratings_df, movies_df_cleaned, on='movie_id')

# Create user-item matrix
user_movie_matrix = merged_df.pivot_table(index='user_id', columns='movie_title', values='rating')

# --- Recommendation Function (Copied from previous steps) ---
def recommend_movies(movie_title, user_movie_matrix, num_recommendations=10):
    try:
        movie_ratings = user_movie_matrix[movie_title]
    except KeyError:
        return pd.Series(dtype=float) # Return empty series for consistency

    similar_to_movie = user_movie_matrix.corrwith(movie_ratings)
    corr_movie = pd.DataFrame(similar_to_movie, columns=['Correlation'])
    corr_movie.dropna(inplace=True)

    movie_ratings_count = user_movie_matrix.count()
    movie_ratings_count_df = pd.DataFrame(movie_ratings_count, columns=['RatingCount'])
    corr_movie = corr_movie.merge(movie_ratings_count_df, left_index=True, right_index=True)

    min_ratings_threshold = 100
    corr_movie = corr_movie[corr_movie['RatingCount'] >= min_ratings_threshold]

    recommendations = corr_movie.sort_values('Correlation', ascending=False)
    recommendations = recommendations[recommendations.index != movie_title]

    return recommendations.head(num_recommendations)

# --- Flask Routes ---
@app.route('/recommend', methods=['GET'])
def get_recommendations():
    movie_title = request.args.get('movie')
    if not movie_title:
        return jsonify({'error': 'Please provide a movie title as a query parameter (e.g., /recommend?movie=Toy Story (1995))'}), 400

    recommendations = recommend_movies(movie_title, user_movie_matrix)

    if recommendations.empty:
        return jsonify({'message': f"No recommendations found for '{movie_title}'. It might not be in the dataset or have enough ratings."}), 404
    else:
        return jsonify(recommendations.to_dict(orient='index'))

@app.route('/')
def home():
    return "<h1>Movie Recommendation API</h1><p>Use /recommend?movie=<movie_title> to get recommendations.</p>"

if __name__ == '__main__':
    # Running on port 5001 now instead of 5000
    app.run(host='127.0.0.1', port=5001)

Overwriting app.py


In [21]:
!pip install pyngrok

In [22]:
from pyngrok import ngrok, conf
import os
import subprocess
import time

# --- IMPORTANT: Replace with your actual ngrok auth token ---
# Get your auth token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "356AcmuLVUYiehIR2JAfxbZlYbo_3XiDLbCbrxgTqhBmNQXdJ"
# --------------------------------------------------------------

# No need for the 'if' check, as the token is already replaced.

try:
    # Ensure any existing ngrok processes are killed
    print("Attempting to kill any existing ngrok processes...")
    ngrok.kill()
    time.sleep(2) # Give a moment for processes to terminate

    # Authenticate ngrok
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Start Flask app in a non-blocking way
    print("Starting Flask app in background...")
    # Use subprocess.Popen to run app.py in a separate process.
    # stdout and stderr are redirected to DEVNULL to prevent clutter in the notebook output.
    flask_process = subprocess.Popen(["python", "app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Give Flask a moment to start up before ngrok tries to connect to it
    time.sleep(5)

    # Start ngrok tunnel
    print("Establishing ngrok tunnel...")
    public_url = ngrok.connect(5000).public_url
    print(f"* ngrok tunnel URL: {public_url}")
    print(f"* Flask app running on: http://127.0.0.1:5000/")
    print(f"* Access your recommendations API at: {public_url}/recommend?movie=Toy%20Story%20(1995)")

except Exception as e:
    print(f"Error starting ngrok or Flask app: {e}")
    print("Make sure your NGROK_AUTH_TOKEN is correct and ngrok is installed/configured.")
    print("If you see 'ERR_NGROK_334' (endpoint already online), try restarting your Colab runtime or waiting a few minutes.")

# Note: The Flask app will now run in the background. It will persist until the Colab runtime is stopped.
# To explicitly stop the Flask app (and ngrok tunnel), you would typically need to restart the runtime
# or add code to kill the `flask_process` (e.g., `flask_process.terminate()`) in a subsequent cell.

Attempting to kill any existing ngrok processes...
Starting Flask app in background...
Establishing ngrok tunnel...
* ngrok tunnel URL: https://unscenically-providential-daleyza.ngrok-free.dev
* Flask app running on: http://127.0.0.1:5000/
* Access your recommendations API at: https://unscenically-providential-daleyza.ngrok-free.dev/recommend?movie=Toy%20Story%20(1995)


In [27]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.7 MB/s eta 0:00:00


Now, let's create a Streamlit application file (`streamlit_app.py`) that will contain our recommendation logic and a simple user interface.

In [32]:
%%writefile streamlit_app.py

import streamlit as st
import pandas as pd
import os
import zipfile
import io
import requests

st.title('Movie Recommendation System')

# --- Data Loading and Preprocessing ---
@st.cache_data # Cache data loading to avoid re-running on every interaction
def load_data():
    MOVIES_URL = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
    DATA_DIR = './ml-100k'

    if not os.path.exists(DATA_DIR):
        # st.write("Downloading MovieLens 100k dataset...") # Removed st.write statement
        response = requests.get(MOVIES_URL)
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            zf.extractall('.')
        # st.write("Download and extraction complete.") # Removed st.write statement

    ratings_cols = ['user_id', 'movie_id', 'rating', 'timestamp']
    ratings_df = pd.read_csv(os.path.join(DATA_DIR, 'u.data'), sep='\t', names=ratings_cols, encoding='latin-1')

    movie_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL',
                  'unknown', 'Action', 'Adventure', 'Animation', 'Children\'s', 'Comedy',
                  'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
                  'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
    movies_df = pd.read_csv(os.path.join(DATA_DIR, 'u.item'), sep='|', names=movie_cols, encoding='latin-1')

    movies_df_cleaned = movies_df.drop(columns=['release_date', 'video_release_date', 'IMDb_URL'])
    ratings_df = ratings_df.drop(columns=['timestamp'])
    merged_df = pd.merge(ratings_df, movies_df_cleaned, on='movie_id')

    user_movie_matrix = merged_df.pivot_table(index='user_id', columns='movie_title', values='rating')
    return user_movie_matrix, movies_df # Return movies_df to get all movie titles

user_movie_matrix, movies_df = load_data()

# --- Recommendation Function ---
@st.cache_data # Cache the recommendation function as well
def recommend_movies(movie_title, user_movie_matrix, num_recommendations=10):
    try:
        movie_ratings = user_movie_matrix[movie_title]
    except KeyError:
        return pd.Series(dtype=float)

    similar_to_movie = user_movie_matrix.corrwith(movie_ratings)
    corr_movie = pd.DataFrame(similar_to_movie, columns=['Correlation'])
    corr_movie.dropna(inplace=True)

    movie_ratings_count = user_movie_matrix.count()
    movie_ratings_count_df = pd.DataFrame(movie_ratings_count, columns=['RatingCount'])
    corr_movie = corr_movie.merge(movie_ratings_count_df, left_index=True, right_index=True)

    min_ratings_threshold = 100
    corr_movie = corr_movie[corr_movie['RatingCount'] >= min_ratings_threshold]

    recommendations = corr_movie.sort_values('Correlation', ascending=False)
    recommendations = recommendations[recommendations.index != movie_title]

    return recommendations.head(num_recommendations)

# --- Streamlit UI ---
st.header('Get Movie Recommendations')

# Get a list of all movie titles for the dropdown
all_movie_titles = sorted(movies_df['movie_title'].unique().tolist())

selected_movie = st.selectbox(
    'Choose a movie:',
    all_movie_titles
)

if st.button('Get Recommendations'):
    if selected_movie:
        st.write(f"## Recommendations for '{selected_movie}':")
        recommendations = recommend_movies(selected_movie, user_movie_matrix)

        if recommendations.empty:
            st.write(f"No recommendations found for '{selected_movie}'. It might not be in the dataset or have enough ratings.")
        else:
            # Display recommendations in a nice format
            st.table(recommendations.reset_index().rename(columns={'index': 'Movie Title', 'Correlation': 'Similarity Score'}))
    else:
        st.write("Please select a movie to get recommendations.")

Overwriting streamlit_app.py


In [30]:
%%writefile requirements.txt
streamlit
pandas
requests

Writing requirements.txt
